
# Гибридная рекомендательная система для фильмов

**Портфолио-проект:** гибридная рекомендательная система на **Python, pandas, NumPy и scikit-learn**.

## Цели проекта
В этом ноутбуке я поставила себе следующие цели:
1. загрузить и подготовить датасет MovieLens;
2. построить **popularity baseline**;
3. строим модель **collaborative filtering** (item-item KNN);
4. объединяем подходы в **гибридную модель**;
5. сравниваем модели по **top-K метрикам**.

## Датасет
В ноутбуке используется официальный датасет **MovieLens latest-small** от GroupLens:  
`https://files.grouplens.org/datasets/movielens/ml-latest-small.zip`

Датасет содержит пользовательские оценки фильмов.  
Чтобы оценивать именно **top-K рекомендации**, мы переводим задачу в формат **implicit feedback**:
- оценка **>= 4.0** → считаем, что пользователю фильм понравился;
- оценка **< 4.0** → не используем как положительное взаимодействие.

In [30]:
import os
import zipfile
import urllib.request
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
np.random.seed(42)



## 1. Загрузка и чтение данных
Следующая ячейка скачивает датасет MovieLens **latest-small**, если его ещё нет локально.


In [31]:
DATA_URL = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
ZIP_FILE = "ml-latest-small.zip"
EXTRACT_DIR = "ml-latest-small"

if not os.path.exists(ZIP_FILE):
    urllib.request.urlretrieve(DATA_URL, ZIP_FILE)

if not os.path.exists(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_FILE, "r") as zip_ref:
        zip_ref.extractall()

ratings = pd.read_csv(f"{EXTRACT_DIR}/ratings.csv")
movies = pd.read_csv(f"{EXTRACT_DIR}/movies.csv")
links = pd.read_csv(f"{EXTRACT_DIR}/links.csv")
tags = pd.read_csv(f"{EXTRACT_DIR}/tags.csv")

print("Размер ratings:", ratings.shape)
print("Размер movies:", movies.shape)
print("Размер links:", links.shape)
print("Размер tags:", tags.shape)

Размер ratings: (100836, 4)
Размер movies: (9742, 3)
Размер links: (9742, 3)
Размер tags: (3683, 4)


In [32]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931



## 2. Краткий обзор датасета
Посмотрим на количество пользователей, фильмов, распределение оценок и общие характеристики данных.


In [33]:

basic_stats = pd.DataFrame(
    {
        "Показатель": [
            "Количество оценок",
            "Количество пользователей",
            "Количество фильмов",
            "Средняя оценка",
            "Медианная оценка",
        ],
        "Значение": [
            len(ratings),
            ratings["userId"].nunique(),
            ratings["movieId"].nunique(),
            ratings["rating"].mean(),
            ratings["rating"].median(),
        ],
    }
)

basic_stats


,Показатель,Значение
0,Количество оценок,100836.000000
1,Количество пользователей,610.000000
2,Количество фильмов,9724.000000
3,Средняя оценка,3.501557
4,Медианная оценка,3.500000


In [34]:

rating_distribution = (
    ratings["rating"]
    .value_counts()
    .sort_index()
    .rename_axis("Оценка")
    .reset_index(name="Количество")
)

rating_distribution


,Оценка,Количество
0,0.5,1370
1,1.0,2811
2,1.5,1791
3,2.0,7551
4,2.5,5550
5,3.0,20047
6,3.5,13136
7,4.0,26818
8,4.5,8551
9,5.0,13211


In [35]:

user_activity = ratings.groupby("userId").size().describe().to_frame("Сводка по числу оценок на пользователя")
movie_activity = ratings.groupby("movieId").size().describe().to_frame("Сводка по числу оценок на фильм")

display(user_activity)
display(movie_activity)


,Сводка по числу оценок на пользователя
count,610.000000
mean,165.304918
std,269.480584
min,20.000000
25%,35.000000
50%,70.500000
75%,168.000000
max,2698.000000


,Сводка по числу оценок на фильм
count,9724.000000
mean,10.369807
std,22.401005
min,1.000000
25%,1.000000
50%,3.000000
75%,9.000000
max,329.000000



## 3. Переход от explicit feedback к implicit feedback

Для top-K рекомендаций удобно рассматривать задачу как задачу ранжирования положительных взаимодействий.

Фильм считается **понравившимся**, если пользователь поставил ему оценку **>= 4.0**.

Далее:
- оставляем только положительные взаимодействия;
- оставляем пользователей минимум с 5 положительными взаимодействиями;
- делаем leave-one-out split:
  - последнее положительное взаимодействие пользователя отправляем в **test**;
  - предыдущие оставляем в **train**.

Это простой и понятный способ оценивать рекомендательные системы.


In [36]:

positive_ratings = ratings[ratings["rating"] >= 4.0].copy()
positive_ratings = positive_ratings.sort_values(["userId", "timestamp"]).reset_index(drop=True)

user_positive_counts = positive_ratings["userId"].value_counts()
eligible_user_ids = user_positive_counts[user_positive_counts >= 5].index

positive_ratings = positive_ratings[positive_ratings["userId"].isin(eligible_user_ids)].copy()

print("Число положительных взаимодействий:", len(positive_ratings))
print("Пользователей с минимум 5 положительными оценками:", positive_ratings["userId"].nunique())
print("Фильмов в положительных взаимодействиях:", positive_ratings["movieId"].nunique())


Число положительных взаимодействий: 48562
Пользователей с минимум 5 положительными оценками: 603
Фильмов в положительных взаимодействиях: 6298


In [50]:

test = positive_ratings.groupby("userId").tail(1).copy()
train = positive_ratings.drop(index=test.index).copy()

train_movie_ids = set(train["movieId"].unique())
test = test[test["movieId"].isin(train_movie_ids)].copy()

eval_user_ids = sorted(test["userId"].unique())
train_user_ids = sorted(train["userId"].unique())

print("Число взаимодействий в train:", len(train))
print("Число взаимодействий в test:", len(test))
print("Число пользователей для оценки:", len(eval_user_ids))


Число взаимодействий в train: 47959
Число взаимодействий в test: 568
Число пользователей для оценки: 568


In [51]:
train.head()

,userId,movieId,rating,timestamp
0,1,804,4.0,964980499
1,1,1210,5.0,964980499
2,1,2018,5.0,964980523
3,1,2628,4.0,964980523
4,1,2826,4.0,964980523



## 4. Подготовка вспомогательных структур и матрицы взаимодействий
Строим бинарную матрицу user-item по train-части данных.


In [52]:

movie_info = movies[["movieId", "title", "genres"]].copy()
movie_title_map = movie_info.set_index("movieId")["title"].to_dict()

train_user_items = train.groupby("userId")["movieId"].apply(set).to_dict()
test_user_items = test.groupby("userId")["movieId"].apply(set).to_dict()

train_user_to_idx = {user_id: idx for idx, user_id in enumerate(sorted(train["userId"].unique()))}
train_movie_to_idx = {movie_id: idx for idx, movie_id in enumerate(sorted(train["movieId"].unique()))}
idx_to_movie = {idx: movie_id for movie_id, idx in train_movie_to_idx.items()}

rows = train["userId"].map(train_user_to_idx)
cols = train["movieId"].map(train_movie_to_idx)
values = np.ones(len(train), dtype=np.float32)

interaction_matrix = csr_matrix(
    (values, (rows, cols)),
    shape=(len(train_user_to_idx), len(train_movie_to_idx)),
)

item_user_matrix = interaction_matrix.T

print("Размер матрицы взаимодействий (users x items):", interaction_matrix.shape)
print("Размер матрицы item-user (items x users):", item_user_matrix.shape)


Размер матрицы взаимодействий (users x items): (603, 6263)
Размер матрицы item-user (items x users): (6263, 603)



## 5. Popularity baseline
Самая простая модель рекомендует самые популярные фильмы из train среди тех, которые пользователь ещё не видел.

Чтобы baseline был чуть устойчивее, учитываю:
- число положительных взаимодействий;
- среднюю оценку фильма в train.


In [53]:

def minmax_scale(series: pd.Series) -> pd.Series:
    series = pd.Series(series).astype(float)
    if series.empty:
        return series
    min_value = series.min()
    max_value = series.max()
    if max_value == min_value:
        return pd.Series(np.ones(len(series)), index=series.index)
    return (series - min_value) / (max_value - min_value)


movie_stats = (
    train.groupby("movieId")
    .agg(
        positive_count=("rating", "size"),
        average_rating=("rating", "mean"),
    )
)

movie_stats["count_score"] = minmax_scale(movie_stats["positive_count"])
movie_stats["rating_score"] = minmax_scale(movie_stats["average_rating"])
movie_stats["popularity_score"] = 0.7 * movie_stats["count_score"] + 0.3 * movie_stats["rating_score"]

movie_stats = movie_stats.sort_values("popularity_score", ascending=False)
popularity_ranking = movie_stats.index.tolist()

movie_stats.head(10)


,positive_count,average_rating,count_score,rating_score,popularity_score
movieId,,,,,
318,270,4.653704,1.000000,0.653704,0.896111
296,242,4.590909,0.895911,0.590909,0.804410
356,247,4.544534,0.914498,0.544534,0.803509
2571,221,4.583710,0.817844,0.583710,0.747604
593,220,4.490909,0.814126,0.490909,0.717161
260,199,4.572864,0.736059,0.572864,0.687101
2959,179,4.589385,0.661710,0.589385,0.640013
527,172,4.630814,0.635688,0.630814,0.634226
1196,166,4.563253,0.613383,0.563253,0.598344



## 6. Collaborative filtering через item-item KNN
Далее я строю **item-based collaborative filtering**:
- каждый фильм представляется вектором пользователей, которым он понравился;
- близость между фильмами считается через **cosine similarity**;
- для конкретного пользователя кандидаты получают баллы от фильмов, похожих на уже понравившиеся ему фильмы.



In [54]:
knn_model = NearestNeighbors(metric="cosine", algorithm="brute", n_neighbors=40)
knn_model.fit(item_user_matrix)

print("KNN-модель обучена.")


KNN-модель обучена.


In [55]:

ALL_TRAIN_MOVIES = list(train_movie_to_idx.keys())


def get_unseen_movies(user_id: int) -> set:
    seen_items = train_user_items.get(user_id, set())
    return set(ALL_TRAIN_MOVIES) - seen_items


def score_cf_candidates(user_id: int, n_neighbors: int = 30) -> pd.Series:
    liked_movies = train_user_items.get(user_id, set())
    if not liked_movies:
        return pd.Series(dtype=float)

    candidate_scores = defaultdict(float)

    for movie_id in liked_movies:
        if movie_id not in train_movie_to_idx:
            continue

        movie_idx = train_movie_to_idx[movie_id]
        distances, indices = knn_model.kneighbors(
            item_user_matrix[movie_idx],
            n_neighbors=n_neighbors + 1,
        )

        for distance, neighbor_idx in zip(distances[0], indices[0]):
            neighbor_movie_id = idx_to_movie[neighbor_idx]

            if neighbor_movie_id in liked_movies:
                continue

            similarity = 1.0 - distance
            if similarity <= 0:
                continue

            candidate_scores[neighbor_movie_id] += similarity

    if not candidate_scores:
        return pd.Series(dtype=float)

    return pd.Series(candidate_scores).sort_values(ascending=False)



## 7. Функции рекомендаций
Реализую три рекомендателя:

1. **Popularity**
2. **Collaborative Filtering**
3. **Hybrid**, который объединяет нормализованные CF- и popularity-оценки


In [56]:

def recommend_popularity(user_id: int, k: int = 10) -> list:
    seen_items = train_user_items.get(user_id, set())
    recommendations = [movie_id for movie_id in popularity_ranking if movie_id not in seen_items]
    return recommendations[:k]


def recommend_cf(user_id: int, k: int = 10, n_neighbors: int = 30) -> list:
    seen_items = train_user_items.get(user_id, set())
    cf_scores = score_cf_candidates(user_id=user_id, n_neighbors=n_neighbors)

    if cf_scores.empty:
        return recommend_popularity(user_id, k=k)

    recommendations = [movie_id for movie_id in cf_scores.index if movie_id not in seen_items][:k]

    if len(recommendations) < k:
        fallback = [
            movie_id
            for movie_id in popularity_ranking
            if movie_id not in seen_items and movie_id not in recommendations
        ]
        recommendations.extend(fallback[: k - len(recommendations)])

    return recommendations


def recommend_hybrid(user_id: int, k: int = 10, alpha: float = 0.7, n_neighbors: int = 30) -> list:
    seen_items = train_user_items.get(user_id, set())
    candidate_movies = list(get_unseen_movies(user_id))

    if not candidate_movies:
        return []

    cf_scores = score_cf_candidates(user_id=user_id, n_neighbors=n_neighbors)
    popularity_scores = movie_stats["popularity_score"]

    score_frame = pd.DataFrame(index=candidate_movies)
    score_frame["cf_score"] = cf_scores.reindex(candidate_movies).fillna(0.0)
    score_frame["popularity_score"] = popularity_scores.reindex(candidate_movies).fillna(0.0)

    score_frame["cf_score_norm"] = minmax_scale(score_frame["cf_score"])
    score_frame["popularity_score_norm"] = minmax_scale(score_frame["popularity_score"])
    score_frame["hybrid_score"] = (
        alpha * score_frame["cf_score_norm"] +
        (1 - alpha) * score_frame["popularity_score_norm"]
    )

    recommendations = (
        score_frame.sort_values("hybrid_score", ascending=False)
        .head(k)
        .index
        .tolist()
    )

    return recommendations


In [57]:

def movies_to_titles(movie_ids: list) -> list:
    return [movie_title_map.get(movie_id, f"movieId={movie_id}") for movie_id in movie_ids]



## 8. Пример рекомендаций для одного пользователя
Перед общей оценкой удобно посмотреть, что именно рекомендуют модели.


In [58]:

example_user_id = eval_user_ids[0]

recent_train_history = (
    train[train["userId"] == example_user_id]
    .sort_values("timestamp", ascending=False)
    .merge(movie_info, on="movieId", how="left")[["movieId", "title", "rating"]]
    .head(10)
)

hidden_test_item = (
    test[test["userId"] == example_user_id]
    .merge(movie_info, on="movieId", how="left")[["movieId", "title", "rating"]]
)

print(f"Пример пользователя: {example_user_id}")
print("\nПоследние положительные фильмы из train:")
display(recent_train_history)

print("\nСкрытый фильм из test:")
display(hidden_test_item)

example_recommendations = pd.DataFrame(
    {
        "Popularity": movies_to_titles(recommend_popularity(example_user_id, k=10)),
        "Collaborative Filtering": movies_to_titles(recommend_cf(example_user_id, k=10)),
        "Hybrid": movies_to_titles(recommend_hybrid(example_user_id, k=10)),
    }
)

example_recommendations


Пример пользователя: 1

Последние положительные фильмы из train:


,movieId,title,rating
0,2012,Back to the Future Part III (1990),4.0
1,2478,¡Three Amigos! (1986),4.0
2,553,Tombstone (1993),5.0
3,157,Canadian Bacon (1995),5.0
4,3053,"Messenger: The Story of Joan of Arc, The (1999)",5.0
5,1298,Pink Floyd: The Wall (1982),5.0
6,3448,"Good Morning, Vietnam (1987)",5.0
7,151,Rob Roy (1995),5.0
8,1224,Henry V (1989),5.0
9,1090,Platoon (1986),4.0



Скрытый фильм из test:


,movieId,title,rating
0,2492,20 Dates (1998),4.0


,Popularity,Collaborative Filtering,Hybrid
0,"Shawshank Redemption, The (1994)","Godfather, The (1972)","Godfather, The (1972)"
1,Pulp Fiction (1994),Pulp Fiction (1994),Pulp Fiction (1994)
2,"Godfather, The (1972)",Ferris Bueller's Day Off (1986),"Shawshank Redemption, The (1994)"
3,"Lord of the Rings: The Fellowship of the Ring,...",Terminator 2: Judgment Day (1991),Terminator 2: Judgment Day (1991)
4,Terminator 2: Judgment Day (1991),"Sixth Sense, The (1999)","Sixth Sense, The (1999)"
5,"Lord of the Rings: The Return of the King, The...",Die Hard (1988),Ferris Bueller's Day Off (1986)
6,"Lord of the Rings: The Two Towers, The (2002)",Aliens (1986),Die Hard (1988)
7,Memento (2000),"Shawshank Redemption, The (1994)",Aliens (1986)
8,"Dark Knight, The (2008)","Godfather: Part II, The (1974)","Godfather: Part II, The (1974)"
9,Apollo 13 (1995),Blade Runner (1982),Blade Runner (1982)



## 9. Оценка качества по top-K метрикам

Оцениваю каждую модель в постановке **leave-one-out**.

### Метрики
Для каждого пользователя считаю:
- **Precision@K**
- **Recall@K**
- **HitRate@K**
- **MAP@K**
- **NDCG@K**

Дополнительно считаю **coverage**:
- долю уникальных фильмов, которые вообще попали в рекомендации модели, относительно числа фильмов в train.


In [59]:

def precision_at_k(recommended: list, relevant: set, k: int) -> float:
    recommended_k = recommended[:k]
    if k == 0:
        return 0.0
    hits = sum(1 for item in recommended_k if item in relevant)
    return hits / k


def recall_at_k(recommended: list, relevant: set, k: int) -> float:
    recommended_k = recommended[:k]
    if not relevant:
        return 0.0
    hits = sum(1 for item in recommended_k if item in relevant)
    return hits / len(relevant)


def hitrate_at_k(recommended: list, relevant: set, k: int) -> float:
    recommended_k = recommended[:k]
    return float(any(item in relevant for item in recommended_k))


def average_precision_at_k(recommended: list, relevant: set, k: int) -> float:
    recommended_k = recommended[:k]
    if not relevant:
        return 0.0

    score = 0.0
    hits = 0

    for rank, item in enumerate(recommended_k, start=1):
        if item in relevant:
            hits += 1
            score += hits / rank

    return score / min(len(relevant), k)


def ndcg_at_k(recommended: list, relevant: set, k: int) -> float:
    recommended_k = recommended[:k]

    dcg = 0.0
    for rank, item in enumerate(recommended_k, start=1):
        if item in relevant:
            dcg += 1 / np.log2(rank + 1)

    ideal_hits = min(len(relevant), k)
    if ideal_hits == 0:
        return 0.0

    idcg = sum(1 / np.log2(rank + 1) for rank in range(1, ideal_hits + 1))
    return dcg / idcg


In [60]:

def evaluate_model(recommend_function, user_ids: list, k: int = 10) -> tuple[pd.DataFrame, float]:
    rows = []
    recommended_catalog = []

    for user_id in user_ids:
        relevant_items = test_user_items.get(user_id, set())
        recommendations = recommend_function(user_id, k=k)

        rows.append(
            {
                "userId": user_id,
                "Precision@K": precision_at_k(recommendations, relevant_items, k),
                "Recall@K": recall_at_k(recommendations, relevant_items, k),
                "HitRate@K": hitrate_at_k(recommendations, relevant_items, k),
                "MAP@K": average_precision_at_k(recommendations, relevant_items, k),
                "NDCG@K": ndcg_at_k(recommendations, relevant_items, k),
            }
        )

        recommended_catalog.extend(recommendations)

    metrics_df = pd.DataFrame(rows)
    coverage = len(set(recommended_catalog)) / len(ALL_TRAIN_MOVIES)

    return metrics_df, coverage


In [61]:
K_VALUES = [5, 10]
MAX_K = max(K_VALUES)

model_registry = {
    "Popularity": recommend_popularity,
    "Collaborative Filtering": recommend_cf,
    "Hybrid": recommend_hybrid,
}

evaluation_rows = []

# заранее считаем рекомендации для каждого пользователя и каждой модели
precomputed_recs = {}

for model_name, recommend_function in model_registry.items():
    precomputed_recs[model_name] = {}
    for user_id in eval_user_ids:
        precomputed_recs[model_name][user_id] = recommend_function(user_id, k=MAX_K)

for model_name in model_registry.keys():
    for k in K_VALUES:
        rows = []
        recommended_catalog = []

        for user_id in eval_user_ids:
            relevant_items = test_user_items.get(user_id, set())
            recommendations = precomputed_recs[model_name][user_id][:k]

            rows.append(
                {
                    "userId": user_id,
                    "Precision@K": precision_at_k(recommendations, relevant_items, k),
                    "Recall@K": recall_at_k(recommendations, relevant_items, k),
                    "HitRate@K": hitrate_at_k(recommendations, relevant_items, k),
                    "MAP@K": average_precision_at_k(recommendations, relevant_items, k),
                    "NDCG@K": ndcg_at_k(recommendations, relevant_items, k),
                }
            )

            recommended_catalog.extend(recommendations)

        metrics_df = pd.DataFrame(rows)
        coverage = len(set(recommended_catalog)) / len(ALL_TRAIN_MOVIES)

        evaluation_rows.append(
            {
                "Модель": model_name,
                "K": k,
                "Precision@K": metrics_df["Precision@K"].mean(),
                "Recall@K": metrics_df["Recall@K"].mean(),
                "HitRate@K": metrics_df["HitRate@K"].mean(),
                "MAP@K": metrics_df["MAP@K"].mean(),
                "NDCG@K": metrics_df["NDCG@K"].mean(),
                "Coverage": coverage,
            }
        )

results_df = (
    pd.DataFrame(evaluation_rows)
    .sort_values(["K", "NDCG@K"], ascending=[True, False])
    .reset_index(drop=True)
)

results_df

,Модель,K,Precision@K,Recall@K,HitRate@K,MAP@K,NDCG@K,Coverage
0,Collaborative Filtering,5,0.006690,0.033451,0.033451,0.015141,0.019608,0.044707
1,Popularity,5,0.006690,0.033451,0.033451,0.015082,0.019577,0.006227
2,Hybrid,5,0.006338,0.031690,0.031690,0.013087,0.017672,0.035606
3,Collaborative Filtering,10,0.006690,0.066901,0.066901,0.019406,0.030224,0.077439
4,Hybrid,10,0.006162,0.061620,0.061620,0.016750,0.027016,0.060833
5,Popularity,10,0.004225,0.042254,0.042254,0.016199,0.022364,0.009420


## 10. Интерпретация результатов

По полученным результатам я заметила, что лучшей моделью в моём эксперименте оказался **Collaborative Filtering**. Именно этот подход показал самые высокие значения `Recall@K`, `HitRate@K` и `NDCG@K`, особенно при `K = 10`. Из этого я сделала вывод, что персонализированные рекомендации на основе похожих фильмов в данном проекте работают лучше, чем рекомендации только по общей популярности.

Также я увидела, что модель **Popularity** действительно может служить хорошим baseline, однако её рекомендации слабо персонализированы. Это особенно заметно по низкому значению `Coverage`: модель рекомендует сравнительно узкий набор одних и тех же популярных фильмов большинству пользователей.

Гибридная модель **Hybrid** показала результат лучше, чем popularity baseline, но всё же уступила чистому collaborative filtering. Я сделала вывод, что сама идея гибридного подхода является полезной, однако в моей реализации она требует дополнительной настройки. Вероятно, выбранный коэффициент объединения компонентов оказался не оптимальным, и popularity-часть в текущем виде немного ослабляет персонализацию.

При повторном запуске кода я получила те же резултаты.

Таким образом, по итогам эксперимента я пришла к выводу, что в рамках данного датасета и выбранной реализации наиболее удачным подходом оказался **Collaborative Filtering**. При этом гибридная модель остаётся перспективной, но для получения лучшего результата её стоит дополнительно доработать и настроить.

In [63]:

results_df.style.format(
    {
        "Precision@K": "{:.4f}",
        "Recall@K": "{:.4f}",
        "HitRate@K": "{:.4f}",
        "MAP@K": "{:.4f}",
        "NDCG@K": "{:.4f}",
        "Coverage": "{:.4f}",
    }
)


,Модель,K,Precision@K,Recall@K,HitRate@K,MAP@K,NDCG@K,Coverage
0,Collaborative Filtering,5,0.0067,0.0335,0.0335,0.0151,0.0196,0.0447
1,Popularity,5,0.0067,0.0335,0.0335,0.0151,0.0196,0.0062
2,Hybrid,5,0.0063,0.0317,0.0317,0.0131,0.0177,0.0356
3,Collaborative Filtering,10,0.0067,0.0669,0.0669,0.0194,0.0302,0.0774
4,Hybrid,10,0.0062,0.0616,0.0616,0.0167,0.0270,0.0608
5,Popularity,10,0.0042,0.0423,0.0423,0.0162,0.0224,0.0094
